In [ ]:
# ========== 1. IMPORTS ========== #
import os
import csv
import time
import random
import shutil
import glob
import gc
import tempfile
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Activation, Input,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, Callback
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    top_k_accuracy_score, roc_curve, auc,
    cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:
# ========== CUSTOM LOSS: ADAPTIVE CLASS-BALANCED FOCAL LOSS ========== #
#
# Novel contribution:
# Standard CB Focal Loss (Cui et al., 2019) uses STATIC weights based on
# training set class frequencies. Our Adaptive version adds a DYNAMIC
# multiplicative factor updated each epoch based on per-class validation recall.
#
# Formula:
#   w_c^(t) = w_c^(static) * adaptation_factor_c^(t)
#   adaptation_factor_c^(t) = (1 - tau) * adaptation_factor_c^(t-1) + tau * (1 - recall_c^(t-1))
#
# Where tau is the smoothing factor (EMA-style) to prevent oscillation.
# Classes with LOW recall get HIGHER weight → model focuses more on them.
#
# References:
#   - Cui et al., "Class-Balanced Loss Based on Effective Number of Samples", CVPR 2019
#   - Lin et al., "Focal Loss for Dense Object Detection", ICCV 2017
#   - Our novel adaptive weighting mechanism

class AdaptiveClassBalancedFocalLoss(tf.keras.losses.Loss):
    """Adaptive Class-Balanced Focal Loss.
    
    Combines:
    1. Class-Balanced weighting (effective number, Cui et al. 2019)
    2. Focal modulation (hard example mining, Lin et al. 2017)
    3. [NOVEL] Epoch-adaptive per-class weight adjustment based on val recall
    
    The adaptive weights are stored as a tf.Variable and updated externally
    by the AdaptiveWeightCallback after each epoch.
    """
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma              = gamma
        self.beta               = beta
        self.num_classes        = num_classes
        self._samples_per_class = list(samples_per_class)
        
        # Static CB weights (baseline)
        n       = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.static_weights = tf.constant(weights, dtype=tf.float32)
        
        # Adaptive factor — initialized to 1.0, updated by callback
        self.adaptive_factor = tf.Variable(
            tf.ones([num_classes], dtype=tf.float32),
            trainable=False, name='adaptive_cb_factor'
        )

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Combined weight = static CB * adaptive factor
        combined_weights = self.static_weights * self.adaptive_factor
        # Normalize to maintain loss scale
        combined_weights = combined_weights / tf.reduce_mean(combined_weights) 
        
        # Per-sample weight based on true class
        sample_w = tf.reduce_sum(y_true * combined_weights, axis=-1)
        
        # Focal modulation
        pt    = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        
        # Cross-entropy
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'samples_per_class': self._samples_per_class,
            'num_classes': self.num_classes,
            'gamma': self.gamma,
            'beta': self.beta,
        })
        return cfg


class AdaptiveWeightCallback(Callback):
    """Updates AdaptiveClassBalancedFocalLoss weights after each epoch.
    
    Mechanism:
    - Evaluates model on validation set after each epoch
    - Computes per-class recall
    - Updates adaptive_factor using EMA:
        factor_c = (1-tau)*factor_c + tau*(1 - recall_c + epsilon)
    - Classes with LOW recall → HIGHER factor → MORE loss weight
    
    Parameters
    ----------
    loss_fn : AdaptiveClassBalancedFocalLoss instance
    val_ds  : validation tf.data.Dataset
    num_classes : int
    tau     : float, smoothing factor (0.1-0.5 recommended)
              Lower tau = more conservative updates (less oscillation)
              Higher tau = faster adaptation (may oscillate)
    """
    def __init__(self, loss_fn, val_ds, num_classes, class_names, tau=0.3, **kwargs):
        super().__init__(**kwargs)
        self.loss_fn     = loss_fn
        self.val_ds      = val_ds
        self.num_classes = num_classes
        self.class_names = class_names
        self.tau         = tau
        self.history     = []  # Track weight evolution
    
    def on_epoch_end(self, epoch, logs=None):
        # Predict on validation set
        y_pred_probs = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        
        # Extract true labels from val_ds (one-hot → int)
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in self.val_ds])
        
        # Compute per-class recall
        per_class_recall = np.zeros(self.num_classes)
        for c in range(self.num_classes):
            mask = y_true == c
            if mask.sum() > 0:
                per_class_recall[c] = (y_pred[mask] == c).mean()
            else:
                per_class_recall[c] = 1.0  # No samples → assume perfect
        
        # Update adaptive factor: classes with LOW recall → HIGHER weight
        # adaptation_target = 1 - recall + epsilon (so perfect class gets ~epsilon weight)
        epsilon = 0.1  # minimum adaptation factor to avoid zero weight
        adaptation_target = (1.0 - per_class_recall) + epsilon
        
        # EMA update
        current_factor = self.loss_fn.adaptive_factor.numpy()
        new_factor = (1.0 - self.tau) * current_factor + self.tau * adaptation_target
        
        # Normalize so mean = 1 (preserve overall loss magnitude)
        new_factor = new_factor / new_factor.mean()
        
        self.loss_fn.adaptive_factor.assign(new_factor.astype(np.float32))
        
        # Log
        self.history.append({
            'epoch': epoch + 1,
            'per_class_recall': per_class_recall.copy(),
            'adaptive_factor': new_factor.copy(),
        })
        
        print(f"  [AdaptiveCB] Epoch {epoch+1} | "
              f"Recall: {dict(zip(self.class_names, [f'{r:.3f}' for r in per_class_recall]))} | "
              f"Factors: {dict(zip(self.class_names, [f'{f:.3f}' for f in new_factor]))}")


print("AdaptiveClassBalancedFocalLoss defined")
print("  - Static CB weights (Cui et al., 2019) as baseline")
print("  - Dynamic adaptation via AdaptiveWeightCallback")
print("  - EMA smoothing (tau) prevents oscillation")
print("  - Per-class recall feedback loop")

In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# ── STRATEGY ──────────────────────────────────────────────────────────────
STRATEGY_KEY    = "finetune_top5_fsda_adaptive_cb"
STRATEGY_LABEL  = "EfficientNetB4 + FSDA + Adaptive CB Loss (Novel)"
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
USE_AUG         = True
LR              = 1e-4
# ──────────────────────────────────────────────────────────────────────────

# ── FSDA HYPERPARAMETERS ──────────────────────────────────────────────────
FSDA_REDUCTION    = 16
FSDA_SPATIAL_KS   = 7
# ──────────────────────────────────────────────────────────────────────────

# ── ADAPTIVE LOSS HYPERPARAMETERS ─────────────────────────────────────────
ADAPTIVE_TAU    = 0.3    # EMA smoothing factor for weight adaptation
FOCAL_GAMMA     = 2.0    # Focal loss gamma
CB_BETA         = 0.9999 # Class-balanced effective number beta
# ──────────────────────────────────────────────────────────────────────────

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (380, 380, 3)
BATCH_SIZE  = 32
EPOCHS      = 30

# --- Shared hyperparameters ---
DROPOUT_RATE    = 0.5
PATIENCE        = 12

# --- Multi-run settings ---
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

# --- Result storage ---
all_runs_results = []

print(f"Strategy   : {STRATEGY_LABEL}  (key={STRATEGY_KEY})")
print(f"Loss       : AdaptiveClassBalancedFocalLoss  (gamma={FOCAL_GAMMA}, beta={CB_BETA}, tau={ADAPTIVE_TAU})")
print(f"Unfreeze   : {UNFREEZE_BLOCKS}  |  Augmentation: {USE_AUG}  |  LR: {LR}")
print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"N_RUNS     : {N_RUNS}  |  Seeds: {RANDOM_SEEDS[:N_RUNS]}")
print(f"FSDA       : reduction={FSDA_REDUCTION}  spatial_ks={FSDA_SPATIAL_KS}")
print(f"XLA JIT    : ON")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Selectively unfreeze EfficientNetB4 blocks (BN always frozen)."""
    base.trainable = False
    if unfreeze_blocks == "all":
        for layer in base.layers:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    elif isinstance(unfreeze_blocks, list) and len(unfreeze_blocks) > 0:
        for layer in base.layers:
            for block_num in unfreeze_blocks:
                if layer.name.startswith(f"block{block_num}"):
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True
                    break
    trainable = sum(1 for l in base.layers if l.trainable)
    total     = len(base.layers)
    print(f"  [{STRATEGY_LABEL}] {trainable}/{total} base layers trainable (BN always frozen)")


def _collect_samples(split_dir, class_to_idx):
    """Walk split_dir -> (abs_paths, int_labels, rel_filenames), sorted."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    """Build GPU-optimised tf.data pipelines (one-hot labels for training)."""
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = efficientnet_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_augmentation=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_augmentation:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True,  apply_augmentation=use_aug)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False, apply_augmentation=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False, apply_augmentation=False)

    cw      = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        class_to_idx      = class_to_idx,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    aug_info = "ON" if use_aug else "OFF"
    print(f"  Augmentation: {aug_info}  |  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds, meta


print("Helper functions defined.")

In [ ]:
# ========== 6. FSDA ARCHITECTURE + MODEL BUILDER ========== #

class FrequencyChannelAttention(tf.keras.layers.Layer):
    """
    Frequency-domain Channel Attention.
    FFT2D → log-magnitude → channel-wise MLP → sigmoid gate.
    All internal computation in float32 for numerical stability.
    """
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        self.fc1 = Dense(r, use_bias=False, dtype='float32', name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32     = tf.cast(x, tf.float32)
        x_t       = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft     = tf.signal.fft2d(x_complex)
        mag       = tf.math.log1p(tf.abs(x_fft))
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])
        attn = tf.nn.relu(self.fc1(freq_desc))
        attn = tf.nn.sigmoid(self.fc2(attn))
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        out  = x_f32 * attn
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg['reduction'] = self.reduction
        return cfg


class FSDABlock(tf.keras.layers.Layer):
    """
    Frequency-Spatial Dual Attention (FSDA) Block.
    Returns: (fused_features, spatial_attn_map)
    """
    def __init__(self, reduction=16, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction      = reduction
        self.spatial_kernel = spatial_kernel

    def build(self, input_shape):
        self.freq_attn = FrequencyChannelAttention(
            self.reduction, name=f'{self.name}_freq_attn')
        self.sp_conv = Conv2D(
            1, self.spatial_kernel, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform', dtype='float32',
            name=f'{self.name}_sp_conv')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        self.freq_attn.build(input_shape)
        sp_input_shape = tuple(input_shape[:-1]) + (2,)
        self.sp_conv.build(sp_input_shape)
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)
        x_f32    = tf.cast(x, tf.float32)
        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32,  axis=-1, keepdims=True)
        sp_attn  = tf.nn.sigmoid(
            self.sp_conv(tf.concat([avg_pool, max_pool], axis=-1))
        )
        spatial_out = x_f32 * sp_attn
        fused = freq_out + spatial_out
        fused = self.bn(fused, training=training)
        fused = tf.cast(fused, input_dtype)
        return fused, sp_attn

    def compute_output_spec(self, x, training=False):
        """Bypass tf.signal.fft2d symbolic tracing for Keras 3."""
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape, dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'spatial_kernel': self.spatial_kernel})
        return cfg


# Custom objects dict for load_model
CUSTOM_OBJECTS = {
    'FrequencyChannelAttention':      FrequencyChannelAttention,
    'FSDABlock':                      FSDABlock,
    'AdaptiveClassBalancedFocalLoss': AdaptiveClassBalancedFocalLoss,
}


def build_fsda_model(input_shape, num_classes, steps_per_epoch, samples_per_class):
    """Build EfficientNetB4 + FSDA + Adaptive CB Focal Loss."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    feat_map = base.output

    attended, sp_attn_map = FSDABlock(
        reduction=FSDA_REDUCTION,
        spatial_kernel=FSDA_SPATIAL_KS,
        name='fsda',
    )(feat_map)

    x   = GlobalAveragePooling2D(name='gap')(attended)
    x   = BatchNormalization(name='head_bn')(x)
    x   = Dense(256, activation='relu', kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x   = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out, name='EfficientNetB4_FSDA_AdaptiveCB')
    vis_model = Model(inputs=base.input, outputs=[out, sp_attn_map],
                      name='EfficientNetB4_FSDA_AdaptiveCB_vis')

    # Build Adaptive CB Focal Loss
    loss_fn = AdaptiveClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes,
        gamma=FOCAL_GAMMA,
        beta=CB_BETA,
    )

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=loss_fn,
        metrics=['accuracy'],
    )
    return model, vis_model, loss_fn


print("FSDA + Adaptive CB Loss architecture defined.")
print("  - FrequencyChannelAttention → which channels carry disease-relevant frequency info")
print("  - SpatialAttention → where in the image are the lesions")
print("  - AdaptiveClassBalancedFocalLoss → dynamic per-class weight adjustment")
print(f"  - Hyperparams: gamma={FOCAL_GAMMA}, beta={CB_BETA}, tau={ADAPTIVE_TAU}")

In [ ]:
# ========== 7. MULTI-RUN TRAINING ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  --  seed={seed}")
    print(f" Strategy : {STRATEGY_LABEL}  |  LR={LR}  |  Aug={USE_AUG}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # ── Build datasets ─────────────────────────────────────────────────────
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    # ── Compute samples per class from training split ──────────────────────
    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
        for cn in meta.class_names
    ])
    print(f"  Samples/class: {dict(zip(meta.class_names, samples_per_class))}")

    # ── Build FSDA model with Adaptive CB Loss ─────────────────────────────
    model, vis_model, loss_fn = build_fsda_model(
        INPUT_SHAPE, meta.num_classes, steps_per_epoch, samples_per_class)
    print(f"  Trainable params: {model.count_params():,}")

    # ── Callbacks ──────────────────────────────────────────────────────────
    adaptive_cb = AdaptiveWeightCallback(
        loss_fn=loss_fn,
        val_ds=val_ds,
        num_classes=meta.num_classes,
        class_names=meta.class_names,
        tau=ADAPTIVE_TAU,
    )

    callbacks = [
        adaptive_cb,
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    # ── Save adaptive weight evolution ─────────────────────────────────────
    weight_history_df = pd.DataFrame([
        {'epoch': h['epoch'],
         **{f'recall_{cn}': h['per_class_recall'][i] for i, cn in enumerate(meta.class_names)},
         **{f'factor_{cn}': h['adaptive_factor'][i] for i, cn in enumerate(meta.class_names)}}
        for h in adaptive_cb.history
    ])
    weight_history_df.to_csv(os.path.join(RESULT_DIR, 'adaptive_weight_history.csv'), index=False)

    # ── Learning curves ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'Accuracy -- Run {run_idx+1}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    # Loss
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'Loss -- Run {run_idx+1}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    # Adaptive factors evolution
    for cn in meta.class_names:
        axes[2].plot(weight_history_df['epoch'], weight_history_df[f'factor_{cn}'], label=cn)
    axes[2].set_title('Adaptive CB Factors'); axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Factor'); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.show()

    # ── Evaluate on test set ───────────────────────────────────────────────
    best_model = load_model(
        os.path.join(RESULT_DIR, 'best_model.keras'),
        custom_objects=CUSTOM_OBJECTS,
    )
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(
            y_true_run, y_pred_run, target_names=class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix -- Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred_run == y_true_run)

    # ── Store results ──────────────────────────────────────────────────────
    all_runs_results.append({
        'run':               run_idx + 1,
        'seed':              seed,
        'accuracy':          test_acc,
        'precision':         report['weighted avg']['precision'],
        'recall':            report['weighted avg']['recall'],
        'f1_score':          report['weighted avg']['f1-score'],
        'per_class_metrics': {c: {'precision': report[c]['precision'],
                                   'recall':    report[c]['recall'],
                                   'f1-score':  report[c]['f1-score']}
                               for c in class_names},
        'result_dir':        RESULT_DIR,
        'history':           history.history,
        'y_true':            y_true_run,
        'y_pred':            y_pred_run,
        'pred_probs':        pred_probs,
        'class_names':       class_names,
        'test_filenames':    meta.test_filenames,
        'n_train':           meta.n_train,
        'n_val':             meta.n_val,
        'n_test':            meta.n_test,
        'adaptive_history':  adaptive_cb.history,
    })

    print(f"\n  Acc={test_acc:.4f}  "
          f"P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  "
          f"F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

# ── Save per-run summary CSV ───────────────────────────────────────────────
summary_path = os.path.join(BASE_RESULT_DIR, 'strategy_summary.csv')
with open(summary_path, 'w', newline='') as csvf:
    fieldnames = ['strategy_key', 'strategy_label', 'unfreeze_blocks', 'lr',
                  'use_aug', 'fsda_reduction', 'fsda_spatial_ks', 'adaptive_tau',
                  'run', 'seed', 'accuracy', 'precision', 'recall', 'f1_score']
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            'strategy_key':    STRATEGY_KEY,
            'strategy_label':  STRATEGY_LABEL,
            'unfreeze_blocks': str(UNFREEZE_BLOCKS),
            'lr':              LR,
            'use_aug':         USE_AUG,
            'fsda_reduction':  FSDA_REDUCTION,
            'fsda_spatial_ks': FSDA_SPATIAL_KS,
            'adaptive_tau':    ADAPTIVE_TAU,
            'run':             r['run'],
            'seed':            r['seed'],
            'accuracy':        r['accuracy'],
            'precision':       r['precision'],
            'recall':          r['recall'],
            'f1_score':        r['f1_score'],
        })

print("\n" + "="*70)
print(f" ALL {N_RUNS} RUN(S) COMPLETED -- {STRATEGY_LABEL}")
print(f" Saved summary -> {summary_path}")
print("="*70)

---
## Section 2 — Results Aggregation & Adaptive Weight Analysis

In [ ]:
# ========== AGGREGATE RESULTS ========== #
print("\n" + "="*80)
print("AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall'] for r in all_runs_results]
f1_scores  = [r['f1_score'] for r in all_runs_results]

overall_stats = {
    'Accuracy':  {'mean': np.mean(accuracies),  'std': np.std(accuracies),  'values': accuracies},
    'Precision': {'mean': np.mean(precisions),  'std': np.std(precisions),  'values': precisions},
    'Recall':    {'mean': np.mean(recalls),     'std': np.std(recalls),     'values': recalls},
    'F1-Score':  {'mean': np.mean(f1_scores),   'std': np.std(f1_scores),   'values': f1_scores},
}

print("OVERALL METRICS:")
print("-" * 60)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
    print(f"              Runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 60)

# Per-class stats
class_names = all_runs_results[0]['class_names']
per_class_stats = {}
for cn in class_names:
    per_class_stats[cn] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][cn][metric] for r in all_runs_results]
        per_class_stats[cn][metric] = {'mean': np.mean(values), 'std': np.std(values), 'values': values}

print("\nPER-CLASS F1-SCORE:")
for cn in class_names:
    s = per_class_stats[cn]['f1-score']
    print(f"  {cn:<30} {s['mean']:.4f} +/- {s['std']:.4f}")

# Additional metrics
print("\nADDITIONAL METRICS:")
for r in all_runs_results:
    kappa   = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc     = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"  Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

In [ ]:
# ========== ADAPTIVE WEIGHT EVOLUTION ANALYSIS ========== #
# This is the KEY visualization for the novel contribution.
# It shows how the loss dynamically adapted to focus on harder classes.

print("\n" + "="*80)
print("ADAPTIVE WEIGHT EVOLUTION ANALYSIS")
print("="*80 + "\n")

n_runs_plot = len(all_runs_results)
fig, axes = plt.subplots(n_runs_plot, 2, figsize=(14, 5 * n_runs_plot))
if n_runs_plot == 1:
    axes = axes[np.newaxis, :]

for run_idx, r in enumerate(all_runs_results):
    adaptive_hist = r['adaptive_history']
    epochs = [h['epoch'] for h in adaptive_hist]
    cn_list = r['class_names']
    
    # Left: Per-class recall over epochs
    for ci, cn in enumerate(cn_list):
        recall_vals = [h['per_class_recall'][ci] for h in adaptive_hist]
        axes[run_idx, 0].plot(epochs, recall_vals, marker='o', markersize=3, label=cn)
    axes[run_idx, 0].set_title(f'Per-Class Val Recall (Run {r["run"]})', fontweight='bold')
    axes[run_idx, 0].set_xlabel('Epoch'); axes[run_idx, 0].set_ylabel('Recall')
    axes[run_idx, 0].legend(); axes[run_idx, 0].grid(alpha=0.3)
    axes[run_idx, 0].set_ylim([0, 1.05])
    
    # Right: Adaptive factors over epochs
    for ci, cn in enumerate(cn_list):
        factor_vals = [h['adaptive_factor'][ci] for h in adaptive_hist]
        axes[run_idx, 1].plot(epochs, factor_vals, marker='s', markersize=3, label=cn)
    axes[run_idx, 1].axhline(1.0, color='black', linestyle='--', lw=1, alpha=0.5)
    axes[run_idx, 1].set_title(f'Adaptive CB Factors (Run {r["run"]})', fontweight='bold')
    axes[run_idx, 1].set_xlabel('Epoch'); axes[run_idx, 1].set_ylabel('Weight Factor')
    axes[run_idx, 1].legend(); axes[run_idx, 1].grid(alpha=0.3)

plt.suptitle('Adaptive Class-Balanced Loss — Weight Evolution\n'
             '(Classes with lower recall get higher weight)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'adaptive_weight_evolution.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved -> adaptive_weight_evolution.png")

# Print final adaptive factors
print("\nFinal Adaptive Factors (last epoch):")
for r in all_runs_results:
    if r['adaptive_history']:
        final = r['adaptive_history'][-1]
        print(f"  Run {r['run']}: {dict(zip(r['class_names'], [f'{f:.3f}' for f in final['adaptive_factor']]))}")

In [ ]:
# ========== COMPARISON: STATIC vs ADAPTIVE WEIGHTS ========== #
# Shows the difference between standard CB weights and our adaptive approach.

print("\nSTATIC vs ADAPTIVE WEIGHT COMPARISON:")
print("="*70)

# Compute static CB weights for reference
samples_per_class_ref = np.array([1050, 306, 704])  # From dataset info
class_names_ref = ['Fully_Peeled_Garlic', 'Partially_Peeled_Garlic', 'Spoiled_Garlic']

eff_num = 1.0 - np.power(CB_BETA, samples_per_class_ref)
static_w = (1.0 - CB_BETA) / eff_num
static_w = static_w / static_w.sum() * len(class_names_ref)

print(f"\nStatic CB Weights (computed once from train distribution):")
for cn, w in zip(class_names_ref, static_w):
    print(f"  {cn:<30} {w:.4f}")

print(f"\nAdaptive CB Weights (final epoch, per run):")
for r in all_runs_results:
    if r['adaptive_history']:
        final_factor = r['adaptive_history'][-1]['adaptive_factor']
        adaptive_w = static_w * final_factor
        adaptive_w = adaptive_w / adaptive_w.mean()
        print(f"  Run {r['run']}:")
        for cn, sw, aw in zip(r['class_names'], static_w, adaptive_w):
            diff = ((aw - sw) / sw) * 100
            print(f"    {cn:<30} static={sw:.4f}  adaptive={aw:.4f}  ({diff:+.1f}%)")

print("\nInterpretation:")
print("  - Positive % → model increased focus on this class (lower recall detected)")
print("  - Negative % → model reduced focus (class already well-classified)")
print("  - This adaptive behavior is the NOVEL CONTRIBUTION of our approach")

In [ ]:
# ========== SAVE COMPREHENSIVE REPORT ========== #

report_lines = []
report_lines.append("=" * 80)
report_lines.append("EfficientNetB4 + FSDA + ADAPTIVE CB LOSS — EXPERIMENT REPORT")
report_lines.append("=" * 80)
report_lines.append(f"Strategy: {STRATEGY_LABEL}")
report_lines.append(f"Key: {STRATEGY_KEY}")
report_lines.append(f"")
report_lines.append(f"NOVEL CONTRIBUTION: Adaptive Class-Balanced Focal Loss")
report_lines.append(f"  - Base: CB Focal Loss (Cui et al., 2019)")
report_lines.append(f"  - Novel: Dynamic per-class weight adaptation based on val recall")
report_lines.append(f"  - Hyperparameters: gamma={FOCAL_GAMMA}, beta={CB_BETA}, tau={ADAPTIVE_TAU}")
report_lines.append(f"")
report_lines.append(f"Architecture: EfficientNetB4 + FSDA Block")
report_lines.append(f"  - FSDA: reduction={FSDA_REDUCTION}, spatial_kernel={FSDA_SPATIAL_KS}")
report_lines.append(f"  - Unfreeze blocks: {UNFREEZE_BLOCKS}")
report_lines.append(f"  - LR: {LR} (ExponentialDecay)")
report_lines.append(f"")
report_lines.append(f"Dataset: {DATA_DIR}")
report_lines.append(f"Runs: {N_RUNS} seeds = {RANDOM_SEEDS[:N_RUNS]}")
report_lines.append(f"")
report_lines.append("OVERALL PERFORMANCE (Mean +/- Std):")
report_lines.append("-" * 60)
for metric, stats in overall_stats.items():
    report_lines.append(f"  {metric:<12}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
report_lines.append(f"")
report_lines.append("PER-CLASS F1-SCORE:")
for cn in class_names:
    s = per_class_stats[cn]['f1-score']
    report_lines.append(f"  {cn:<30} {s['mean']:.4f} +/- {s['std']:.4f}")

report_text = "\n".join(report_lines)
print(report_text)

with open(os.path.join(BASE_RESULT_DIR, 'EXPERIMENT_REPORT.txt'), 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f"\nReport saved -> {BASE_RESULT_DIR}/EXPERIMENT_REPORT.txt")

In [ ]:
# ========== ZIP ALL RESULTS ========== #

zip_output_path = f"/kaggle/working/EfficientNetB4_FSDA_AdaptiveCB_Complete"
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"Archive: {zip_output_path}.zip  ({zip_size:.2f} MB)")
print("DONE!")